# Notebook 07: Transfer Learning

---

## Overview

This notebook implements **transfer learning** using pre-trained models for multi-label chest X-ray disease classification.

**Objectives:**
1. Load pre-trained models (ResNet50, DenseNet121, EfficientNetB3)
2. Adapt for multi-label classification (14 diseases)
3. Implement two-stage training: feature extraction → fine-tuning
4. Compare transfer learning vs custom CNN vs baseline models
5. Select and save best performing model

**Why Transfer Learning?**

Pre-trained models learned from **1.4 million ImageNet images**:
- Low-level features (edges, textures) transfer well to medical imaging
- Saves weeks of training time
- Often outperforms custom CNNs when data is limited

**Outputs:**
- Fine-tuned models for ResNet50, DenseNet121, EfficientNetB3
- Performance comparison across all approaches
- Best model selection for deployment

---

## 💡 Transfer Learning Explained

### The Core Concept

**Analogy:** Learning to ride a motorcycle after knowing how to ride a bicycle
- You don't start from scratch (balance, steering already learned)
- You adapt existing skills to new context (throttle control, gears)
- Much faster than learning motorcycle riding with no prior experience

**In Deep Learning:**

```
Traditional Approach (Notebook 06):
┌──────────────────────────────────────┐
│ Train CNN from scratch               │
│ - Random initialization              │
│ - Learn ALL features from our data  │
│ - Requires large dataset             │
│ - Slow training (weeks on GPU)      │
└──────────────────────────────────────┘

Transfer Learning (This Notebook):
┌──────────────────────────────────────┐
│ Use pre-trained model (ImageNet)    │
│ - Already knows edges, textures     │
│ - Adapt to medical imaging          │
│ - Works with smaller datasets       │
│ - Fast training (hours on GPU)      │
└──────────────────────────────────────┘
```

---

### What is ImageNet?

**ImageNet** is a massive dataset used to train foundation models:
- **1.4 million images** across 1,000 categories
- Natural images: dogs, cats, cars, trees, buildings, etc.
- NOT medical images!

**So why does it help with X-rays?**

Because **low-level features are universal**:
- **Early layers** learn: edges, corners, textures, gradients
  - These exist in BOTH natural images AND X-rays!
- **Middle layers** learn: shapes, patterns, object parts
  - Also transferable (e.g., circular shapes, linear structures)
- **Late layers** learn: specific to original task (dogs vs cats)
  - We REPLACE these with X-ray disease detection

**Real-world example:**
A model trained to detect cat whiskers has learned edge detection. Those same edges help detect rib outlines in X-rays!

---

### Two-Stage Training Strategy

**Stage 1: Feature Extraction**
```python
# Freeze pre-trained layers (don't update them)
base_model.trainable = False

# Only train new classification layers
model.fit(...)
```
- Use ImageNet features as-is
- Only train custom top layers for disease classification
- Fast (few epochs)

**Stage 2: Fine-Tuning**
```python
# Unfreeze last few layers
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False  # Keep early layers frozen

# Train with lower learning rate
model.compile(optimizer=Adam(lr=0.0001))
model.fit(...)
```
- Slightly adjust pre-trained weights
- Adapt features to medical imaging
- Careful (low LR) to avoid destroying learned features

---

### Comparison to Traditional ML

**Transfer learning has NO direct equivalent in traditional tabular ML!**

Closest analogy:
- **Domain adaptation**: Training on one dataset, applying to related dataset
- **Feature reuse**: Using features engineered for one problem on another

But transfer learning is UNIQUE to deep learning because:
- Neural networks learn hierarchical representations
- Early layers learn universal patterns
- Can literally copy billions of pre-trained parameters

**Why it works so well:**
- Like having 1.4 million "warm-up" examples for free
- Model already "knows" what edges and textures look like
- We just teach it what chest X-ray diseases look like

---

## 1. Setup and Configuration

In [ ]:
# Import libraries
import json
import sys
import warnings
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import (
    ResNet50,
    DenseNet121,
    EfficientNetB3,
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Configuration
warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print("✓ Libraries imported successfully")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU'))} GPUs")

In [ ]:
# Define paths
current_path = Path.cwd()

if current_path.name == 'jupyter_notebooks':
    PROJECT_ROOT = current_path.parent
elif (current_path / 'setup.py').exists() or (current_path / 'README.md').exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models' / 'saved_models'

print(f"Project root: {PROJECT_ROOT}")
print(f"Models directory: {MODELS_DIR}")

In [ ]:
# Transfer Learning Configuration
CONFIG = {
    # Image parameters
    'img_height': 224,
    'img_width': 224,
    'channels': 3,
    
    # Training parameters - Stage 1 (Feature extraction)
    'batch_size': 32,
    'epochs_stage1': 10,  # Feature extraction
    'learning_rate_stage1': 0.001,
    
    # Training parameters - Stage 2 (Fine-tuning)
    'epochs_stage2': 20,  # Fine-tuning
    'learning_rate_stage2': 0.0001,  # Lower LR for fine-tuning
    'unfreeze_layers': 20,  # Number of layers to unfreeze from top
    
    # Model architecture
    'dense_units': 512,
    'dropout_rate': 0.5,
    
    # Callbacks
    'early_stopping_patience': 10,
    'reduce_lr_patience': 5,
    
    # Data
    'num_classes': 14,
    'use_sample': True,
    'sample_size': 5000,
    
    'random_state': 42
}

print("Transfer Learning Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 2. Load Data

In [ ]:
# Load split files
train_df = pd.read_csv(PROCESSED_DIR / 'train_split.csv')
val_df = pd.read_csv(PROCESSED_DIR / 'val_split.csv')
test_df = pd.read_csv(PROCESSED_DIR / 'test_split.csv')

# Load preprocessing config
with open(PROCESSED_DIR / 'preprocessing_config.json', 'r') as f:
    prep_config = json.load(f)

disease_classes = prep_config['disease_classes']

print(f"✓ Loaded data splits and configuration")
print(f"  Disease classes: {len(disease_classes)}")

In [ ]:
# Sample if configured
if CONFIG['use_sample']:
    sample_size = CONFIG['sample_size']
    train_df = train_df.sample(n=min(sample_size, len(train_df)), random_state=42)
    val_df = val_df.sample(n=min(sample_size // 5, len(val_df)), random_state=42)
    test_df = test_df.sample(n=min(sample_size // 5, len(test_df)), random_state=42)
    
    print(f"⚠️ Using sample mode:")
    print(f"  Train: {len(train_df):,} images")
    print(f"  Val:   {len(val_df):,} images")
    print(f"  Test:  {len(test_df):,} images")

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

## 3. Data Generators

In [ ]:
# Data generators (same as notebook 06)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='full_path',
    y_col=disease_classes,
    target_size=(CONFIG['img_height'], CONFIG['img_width']),
    batch_size=CONFIG['batch_size'],
    class_mode='raw',
    shuffle=True,
    seed=CONFIG['random_state']
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='full_path',
    y_col=disease_classes,
    target_size=(CONFIG['img_height'], CONFIG['img_width']),
    batch_size=CONFIG['batch_size'],
    class_mode='raw',
    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='full_path',
    y_col=disease_classes,
    target_size=(CONFIG['img_height'], CONFIG['img_width']),
    batch_size=CONFIG['batch_size'],
    class_mode='raw',
    shuffle=False
)

print("✓ Data generators ready")

## 4. Build Transfer Learning Model

### 📚 Model Architecture

We'll use a **functional approach** to build transfer learning models:

```
Input (224×224×3)
    ↓
┌─────────────────────────────────┐
│ Pre-trained Base Model          │
│ (ResNet50 / DenseNet / EfficientNet) │
│                                 │
│ Trained on ImageNet             │
│ - Early layers: edges, textures │
│ - Middle layers: shapes, parts  │
│ - Late layers: [TO BE REPLACED] │
│                                 │
│ Weights: FROZEN initially       │
└─────────────────────────────────┘
    ↓
GlobalAveragePooling2D  ← Condense spatial info
    ↓
Dense(512) + ReLU + Dropout(0.5)  ← NEW layer
    ↓
Dense(14) + Sigmoid  ← NEW layer (multi-label)
    ↓
14 disease probabilities
```

**Key differences from custom CNN:**
- Base: Pre-trained (not random init)
- Weights: Start frozen, then fine-tune
- Training: Two stages instead of one

---

In [ ]:
def build_transfer_model(base_model_class, model_name, input_shape, num_classes, config):
    """
    Build transfer learning model with pre-trained base.
    
    Args:
        base_model_class: Keras application (ResNet50, DenseNet121, etc.)
        model_name: String name for logging
        input_shape: (height, width, channels)
        num_classes: Number of output classes
        config: Configuration dict
    
    Returns:
        Compiled Keras model
    """
    # Load pre-trained base model (without top classification layer)
    base_model = base_model_class(
        weights='imagenet',
        include_top=False,  # Remove ImageNet classification layer
        input_shape=input_shape
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Build complete model
    inputs = keras.Input(shape=input_shape)
    
    # Pre-trained base
    x = base_model(inputs, training=False)
    
    # Custom top layers for chest X-ray classification
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(config['dense_units'], activation='relu')(x)
    x = layers.Dropout(config['dropout_rate'])(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    
    model = keras.Model(inputs, outputs, name=model_name)
    
    return model, base_model


print("✓ Transfer learning model builder defined")

## 5. Model 1: ResNet50

### 📚 About ResNet50

**ResNet** = Residual Network (introduced 2015, won ImageNet competition)

**Key innovation: Skip connections**
```
Input → Conv → Conv → Add(Input) → Output
  ↓___________________↑
     Skip connection
```

**Why skip connections?**
- Solves "vanishing gradient" problem in very deep networks
- Allows training 50+ layer networks (previous limit: ~20 layers)
- Identity mapping: easier to learn "do nothing" than force learning

**Architecture:**
- 50 layers deep
- ~25 million parameters
- Uses bottleneck blocks (1×1 → 3×3 → 1×1 convolutions)

**When to use:**
- Good all-rounder for many tasks
- Moderate size (not too big)
- Well-tested in medical imaging

---

### Stage 1: Feature Extraction

In [ ]:
print("\n" + "="*60)
print("MODEL 1: ResNet50")
print("="*60)

# Build model
input_shape = (CONFIG['img_height'], CONFIG['img_width'], CONFIG['channels'])
resnet_model, resnet_base = build_transfer_model(
    ResNet50, 'resnet50_transfer', input_shape, CONFIG['num_classes'], CONFIG
)

print(f"\nResNet50 architecture:")
print(f"  Base layers: {len(resnet_base.layers)} (frozen)")
print(f"  Total parameters: {resnet_model.count_params():,}")
print(f"  Trainable parameters: {sum([tf.size(w).numpy() for w in resnet_model.trainable_weights]):,}")

In [ ]:
# Compile for Stage 1
resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate_stage1']),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

print("\n" + "-"*60)
print("STAGE 1: Feature Extraction (base frozen)")
print("-"*60)
print(f"Learning rate: {CONFIG['learning_rate_stage1']}")
print(f"Epochs: {CONFIG['epochs_stage1']}")
print("\nTraining top layers only...\n")

# Train Stage 1
history_resnet_s1 = resnet_model.fit(
    train_generator,
    epochs=CONFIG['epochs_stage1'],
    validation_data=val_generator,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
    ],
    verbose=1
)

print("\n✓ Stage 1 complete")

### Stage 2: Fine-Tuning

In [ ]:
print("\n" + "-"*60)
print("STAGE 2: Fine-Tuning (unfreeze last layers)")
print("-"*60)

# Unfreeze last N layers of base model
resnet_base.trainable = True
for layer in resnet_base.layers[:-CONFIG['unfreeze_layers']]:
    layer.trainable = False

trainable_layers = sum([1 for layer in resnet_base.layers if layer.trainable])
print(f"Unfrozen layers: {trainable_layers} / {len(resnet_base.layers)}")
print(f"Trainable parameters: {sum([tf.size(w).numpy() for w in resnet_model.trainable_weights]):,}")

# Recompile with lower learning rate
resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate_stage2']),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

print(f"Learning rate: {CONFIG['learning_rate_stage2']} (reduced 10×)")
print(f"Epochs: {CONFIG['epochs_stage2']}")
print("\nFine-tuning...\n")

# Train Stage 2
history_resnet_s2 = resnet_model.fit(
    train_generator,
    epochs=CONFIG['epochs_stage2'],
    validation_data=val_generator,
    callbacks=[
        callbacks.ModelCheckpoint(
            str(MODELS_DIR / 'resnet50_transfer_best.h5'),
            monitor='val_auc',
            mode='max',
            save_best_only=True
        ),
        callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=10, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    ],
    verbose=1
)

print("\n✓ ResNet50 training complete")

## 6. Model 2: DenseNet121

### 📚 About DenseNet121

**DenseNet** = Densely Connected Network (introduced 2017)

**Key innovation: Dense connections**
```
Layer 1 output → Concat → Layer 2 → Concat → Layer 3 → Concat → Layer 4
    ↓____________↑           ↓_______________↑            ↓
    ↓____________________________________↑                ↓
    ↓_________________________________________________________↑
```

Every layer receives inputs from ALL previous layers!

**Why dense connections?**
- Maximum information flow between layers
- Feature reuse (don't learn same features multiple times)
- Fewer parameters than ResNet (more efficient)
- Strong gradient flow (helps training)

**Architecture:**
- 121 layers (deep!)
- ~8 million parameters (despite being deeper than ResNet50!)
- Uses "growth rate" to control width

**When to use:**
- Want parameter efficiency
- Limited GPU memory
- Often performs better than ResNet on medical images

---

In [ ]:
print("\n" + "="*60)
print("MODEL 2: DenseNet121")
print("="*60)

# Build model
densenet_model, densenet_base = build_transfer_model(
    DenseNet121, 'densenet121_transfer', input_shape, CONFIG['num_classes'], CONFIG
)

print(f"\nDenseNet121 architecture:")
print(f"  Base layers: {len(densenet_base.layers)}")
print(f"  Total parameters: {densenet_model.count_params():,}")

# Compile for Stage 1
densenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate_stage1']),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

print("\n" + "-"*60)
print("STAGE 1: Feature Extraction")
print("-"*60)

# Train Stage 1
history_densenet_s1 = densenet_model.fit(
    train_generator,
    epochs=CONFIG['epochs_stage1'],
    validation_data=val_generator,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
    ],
    verbose=1
)

# Stage 2: Fine-tuning
print("\n" + "-"*60)
print("STAGE 2: Fine-Tuning")
print("-"*60)

densenet_base.trainable = True
for layer in densenet_base.layers[:-CONFIG['unfreeze_layers']]:
    layer.trainable = False

densenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate_stage2']),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

history_densenet_s2 = densenet_model.fit(
    train_generator,
    epochs=CONFIG['epochs_stage2'],
    validation_data=val_generator,
    callbacks=[
        callbacks.ModelCheckpoint(
            str(MODELS_DIR / 'densenet121_transfer_best.h5'),
            monitor='val_auc',
            mode='max',
            save_best_only=True
        ),
        callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=10, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    ],
    verbose=1
)

print("\n✓ DenseNet121 training complete")

## 7. Model 3: EfficientNetB3

### 📚 About EfficientNet

**EfficientNet** = Efficient Network (introduced 2019, Google)

**Key innovation: Compound scaling**

Traditional approach: Scale one dimension
- Make network deeper (more layers)
- OR make it wider (more filters)
- OR use larger images

**EfficientNet approach: Scale ALL dimensions together**
```
depth × width × resolution = constant
```

Find optimal balance using neural architecture search (NAS).

**Architecture:**
- B0-B7 variants (B0 = small, B7 = large)
- B3: ~12 million parameters
- Uses mobile inverted bottleneck blocks
- Squeeze-and-excitation (SE) blocks for channel attention

**When to use:**
- Best accuracy per parameter
- State-of-art results on many benchmarks
- Good for production (efficient inference)

**Fun fact:** EfficientNetB7 achieved 84.4% top-1 accuracy on ImageNet with 66M parameters. Previous best (GPipe) needed 556M parameters for 84.3%!

---

In [ ]:
print("\n" + "="*60)
print("MODEL 3: EfficientNetB3")
print("="*60)

# Build model
efficientnet_model, efficientnet_base = build_transfer_model(
    EfficientNetB3, 'efficientnetb3_transfer', input_shape, CONFIG['num_classes'], CONFIG
)

print(f"\nEfficientNetB3 architecture:")
print(f"  Base layers: {len(efficientnet_base.layers)}")
print(f"  Total parameters: {efficientnet_model.count_params():,}")

# Compile for Stage 1
efficientnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate_stage1']),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

print("\n" + "-"*60)
print("STAGE 1: Feature Extraction")
print("-"*60)

# Train Stage 1
history_efficientnet_s1 = efficientnet_model.fit(
    train_generator,
    epochs=CONFIG['epochs_stage1'],
    validation_data=val_generator,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
    ],
    verbose=1
)

# Stage 2: Fine-tuning
print("\n" + "-"*60)
print("STAGE 2: Fine-Tuning")
print("-"*60)

efficientnet_base.trainable = True
for layer in efficientnet_base.layers[:-CONFIG['unfreeze_layers']]:
    layer.trainable = False

efficientnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate_stage2']),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

history_efficientnet_s2 = efficientnet_model.fit(
    train_generator,
    epochs=CONFIG['epochs_stage2'],
    validation_data=val_generator,
    callbacks=[
        callbacks.ModelCheckpoint(
            str(MODELS_DIR / 'efficientnetb3_transfer_best.h5'),
            monitor='val_auc',
            mode='max',
            save_best_only=True
        ),
        callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=10, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    ],
    verbose=1
)

print("\n✓ EfficientNetB3 training complete")

## 8. Evaluate All Models on Test Set

In [ ]:
# Load best saved models
resnet_best = keras.models.load_model(MODELS_DIR / 'resnet50_transfer_best.h5')
densenet_best = keras.models.load_model(MODELS_DIR / 'densenet121_transfer_best.h5')
efficientnet_best = keras.models.load_model(MODELS_DIR / 'efficientnetb3_transfer_best.h5')

print("✓ Loaded best models from checkpoints")

In [ ]:
# Evaluate all models
print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)

results_dict = {}

for model_name, model in [
    ('ResNet50', resnet_best),
    ('DenseNet121', densenet_best),
    ('EfficientNetB3', efficientnet_best)
]:
    print(f"\n{model_name}:")
    test_results = model.evaluate(test_generator, verbose=0)
    
    results_dict[model_name] = {
        'loss': float(test_results[0]),
        'accuracy': float(test_results[1]),
        'auc': float(test_results[2])
    }
    
    print(f"  Loss: {test_results[0]:.4f}")
    print(f"  Accuracy: {test_results[1]:.4f}")
    print(f"  AUC: {test_results[2]:.4f}")

print("\n" + "="*60)

## 9. Compare with All Previous Approaches

In [ ]:
# Load baseline and CNN results
with open(OUTPUTS_DIR / 'reports' / '05_baseline_models_results.json', 'r') as f:
    baseline_results = json.load(f)

with open(OUTPUTS_DIR / 'reports' / '06_cnn_results.json', 'r') as f:
    cnn_results = json.load(f)

# Create comparison table
comparison_data = {
    'Approach': [],
    'Model': [],
    'Test AUC': [],
    'Parameters': []
}

# Baseline models
comparison_data['Approach'].append('Baseline (Hand-crafted features)')
comparison_data['Model'].append(baseline_results['best_model']['name'])
comparison_data['Test AUC'].append(baseline_results['best_model']['test_avg_auc'])
comparison_data['Parameters'].append('~1.8K features')

# Custom CNN
comparison_data['Approach'].append('Custom CNN')
comparison_data['Model'].append('4-block CNN')
comparison_data['Test AUC'].append(cnn_results['test_performance']['overall']['auc'])
comparison_data['Parameters'].append(f"{cnn_results['architecture']['total_params']:,}")

# Transfer learning models
for model_name in ['ResNet50', 'DenseNet121', 'EfficientNetB3']:
    comparison_data['Approach'].append('Transfer Learning')
    comparison_data['Model'].append(model_name)
    comparison_data['Test AUC'].append(results_dict[model_name]['auc'])
    
    if model_name == 'ResNet50':
        comparison_data['Parameters'].append('~25M')
    elif model_name == 'DenseNet121':
        comparison_data['Parameters'].append('~8M')
    else:
        comparison_data['Parameters'].append('~12M')

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*60)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*60)
print(comparison_df.to_string(index=False))
print("\n" + "="*60)

# Find best model
best_idx = comparison_df['Test AUC'].idxmax()
best_model_info = comparison_df.iloc[best_idx]

print(f"\n🏆 BEST MODEL: {best_model_info['Model']}")
print(f"   Approach: {best_model_info['Approach']}")
print(f"   Test AUC: {best_model_info['Test AUC']:.4f}")
print(f"   Parameters: {best_model_info['Parameters']}")

## 10. Visualize Comparison

In [ ]:
# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#e74c3c', '#3498db', '#2ecc71', '#2ecc71', '#2ecc71']
bars = ax.barh(comparison_df['Model'], comparison_df['Test AUC'], color=colors, alpha=0.8)

# Add value labels
for i, (bar, auc) in enumerate(zip(bars, comparison_df['Test AUC'])):
    ax.text(auc + 0.005, bar.get_y() + bar.get_height()/2, 
            f'{auc:.3f}', va='center', fontweight='bold')

ax.set_xlabel('Test AUC', fontweight='bold', fontsize=12)
ax.set_title('Model Performance Comparison: Baseline → CNN → Transfer Learning', 
             fontweight='bold', fontsize=14)
ax.set_xlim([0, 1])
ax.grid(axis='x', alpha=0.3)

# Add approach labels
for i, (model, approach) in enumerate(zip(comparison_df['Model'], comparison_df['Approach'])):
    ax.text(-0.02, i, approach, ha='right', va='center', 
            fontsize=9, style='italic', color='gray')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '07_transfer_learning_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved comparison visualization")

## 11. Save Results

In [ ]:
# Compile all results
transfer_learning_results = {
    'config': CONFIG,
    'models': {
        'resnet50': results_dict['ResNet50'],
        'densenet121': results_dict['DenseNet121'],
        'efficientnetb3': results_dict['EfficientNetB3']
    },
    'best_model': {
        'name': best_model_info['Model'],
        'approach': best_model_info['Approach'],
        'test_auc': float(best_model_info['Test AUC']),
        'parameters': best_model_info['Parameters']
    },
    'comparison': comparison_df.to_dict('records')
}

# Save to JSON
with open(OUTPUTS_DIR / 'reports' / '07_transfer_learning_results.json', 'w') as f:
    json.dump(transfer_learning_results, f, indent=2)

print(f"✓ Saved results to {OUTPUTS_DIR / 'reports' / '07_transfer_learning_results.json'}")

## 12. Summary

In [ ]:
print("="*60)
print("  ✅ Notebook 07 Complete: Transfer Learning")
print("="*60)

print("\n🏗️ Models Trained:")
print("  1. ResNet50 (25M params)")
print("  2. DenseNet121 (8M params)")
print("  3. EfficientNetB3 (12M params)")

print("\n📊 Training Strategy:")
print(f"  Stage 1: Feature extraction ({CONFIG['epochs_stage1']} epochs, base frozen)")
print(f"  Stage 2: Fine-tuning ({CONFIG['epochs_stage2']} epochs, last {CONFIG['unfreeze_layers']} layers unfrozen)")

print("\n🎯 Test Performance:")
for model_name in ['ResNet50', 'DenseNet121', 'EfficientNetB3']:
    print(f"  {model_name}: AUC = {results_dict[model_name]['auc']:.4f}")

print(f"\n🏆 Best Overall Model: {best_model_info['Model']}")
print(f"   Test AUC: {best_model_info['Test AUC']:.4f}")

print("\n📈 Performance Evolution:")
baseline_auc = baseline_results['best_model']['test_avg_auc']
custom_cnn_auc = cnn_results['test_performance']['overall']['auc']
best_transfer_auc = best_model_info['Test AUC']

print(f"  Baseline → CNN: {((custom_cnn_auc - baseline_auc) / baseline_auc * 100):+.1f}%")
print(f"  CNN → Transfer: {((best_transfer_auc - custom_cnn_auc) / custom_cnn_auc * 100):+.1f}%")
print(f"  Baseline → Best: {((best_transfer_auc - baseline_auc) / baseline_auc * 100):+.1f}%")

print("\n📁 Generated Files:")
print(f"  {MODELS_DIR / 'resnet50_transfer_best.h5'}")
print(f"  {MODELS_DIR / 'densenet121_transfer_best.h5'}")
print(f"  {MODELS_DIR / 'efficientnetb3_transfer_best.h5'}")
print(f"  {OUTPUTS_DIR / 'reports' / '07_transfer_learning_results.json'}")
print(f"  {FIGURES_DIR / '07_transfer_learning_comparison.png'}")

print("\n💡 Key Insights:")
print("  - Transfer learning leverages 1.4M ImageNet images")
print("  - Two-stage training: feature extraction → fine-tuning")
print("  - Pre-trained models outperform custom CNN (limited data scenario)")
print("  - DenseNet121 most parameter-efficient (8M vs 25M ResNet)")

print("\n⏭️  Next: Notebook 08 - Model Evaluation & Interpretation!")
print("  Grad-CAM visualizations, per-disease ROC curves, error analysis")